# Day 5 — Solution: Heteroskedasticity & White's Repair

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - X.shape[1])
    se = np.sqrt(np.diag(s2 * XtX_inv))
    return {"alpha": b[0], "beta": b[1], "se_beta": se[1], "resid": e, "X": X}

def se_white(X, e):
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = X.T @ (e[:, None] ** 2 * X)
    return np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv))

## E1 — the sandwich, by hand vs statsmodels

In [ ]:
rng = np.random.default_rng(11)
n = 1200
m = rng.normal(0, 0.011, n)
sig = 0.006 * (0.4 + 3.5 * np.abs(m) / 0.011)
y = 0.0001 + 0.9 * m + rng.normal(0, 1, n) * sig

f = ols_fit(m, y)
hc0 = se_white(f["X"], f["resid"])[1]
k = 2
hc1 = hc0 * np.sqrt(n / (n - k))
import statsmodels.api as sm
se_sm = np.asarray(sm.OLS(y, sm.add_constant(m)).fit(cov_type="HC1").bse)[1]
print(f"classical SE {f['se_beta']:.4f}")
print(f"White HC0    {hc0:.4f}")
print(f"White HC1    {hc1:.4f}   statsmodels HC1 {se_sm:.4f}   match {np.isclose(hc1, se_sm)}")

**Expected reasoning.** HC1 matches statsmodels to precision — again: your
meat-and-bread *is* the library's. The interesting column is classical vs
White on THIS draw: the White SE is ~1.3–1.6× the classical here (variance
loads on |m|, and those points drive β̂). **A fixed multiple on one draw is
noise; the systematic gap across 600 draws is E2's business.** Also note
what did *not* happen: β̂ itself is unchanged to the last decimal — the
sandwich reprices the error bar, never the estimate.

## E2 — the size tournament, repaired

In [ ]:
n_runs, n = 600, 1200
est = np.empty(n_runs); se_c = np.empty(n_runs); se_w = np.empty(n_runs)
for i in range(n_runs):
    rng = np.random.default_rng(9000 + i)
    m = rng.normal(0, 0.011, n)
    sig = 0.006 * (0.4 + 3.5 * np.abs(m) / 0.011)
    y = 0.0001 + 0.9 * m + rng.normal(0, 1, n) * sig
    f = ols_fit(m, y)
    est[i] = f["beta"]; se_c[i] = f["se_beta"]
    se_w[i] = se_white(f["X"], f["resid"])[1] * np.sqrt(n / (n - 2))

sd_true = est.std(ddof=1)
for name, se in [("classical", se_c), ("White HC1", se_w)]:
    rej = (np.abs(est - 0.9) / se > 1.96).mean()
    print(f"{name:10s}: mean SE {se.mean():.4f} vs true sd {sd_true:.4f}  -> rejection {rej:.3f} (nominal 0.05)")

**Expected numbers.** True sd of β̂ ≈ 0.09–0.10 — the heteroskedasticity
premium is ~60% on top of the classical read. Classical: mean SE ≈ 0.06 →
rejection ≈ 0.20–0.22: **a nominally-5% test that fires ~4× too often, on
a null that is TRUE.** White HC1: mean SE ≈ 0.10, tracking the true sd →
rejection ≈ 0.04–0.07. The day's entire
argument in two rows: wrong SE manufactures t-stats; the sandwich holds
size *without ever modeling the variance process*. File the corollary for
module 13: a literature average t-stat inflated by a wrong-SE factor of
1.5× needs no p-hacking to look publishable.

## E3 — the fan, on real data

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    rr = np.log(get_prices(["SPY", "QQQ"], start="2010-01-01")).diff().dropna()
else:
    p = synthetic_prices(n_days=3800, n_assets=2, seed=5, corr=0.85)
    rr = np.log(p).diff().dropna(); rr.columns = ["SPY", "QQQ"]

f = ols_fit(rr["SPY"].values, rr["QQQ"].values)
x, e = rr["SPY"].values, f["resid"]
q = pd.qcut(np.abs(x), 4, labels=False)
for i in range(4):
    sel = q == i
    print(f"|x| quartile {i+1}: mean|e| {np.abs(e[sel]).mean():.5f}   var(e) {e[sel].var():.2e}")

aux = sm.OLS(e**2, sm.add_constant(np.column_stack([x, x**2]))).fit()
print(f"White test aux: t(x) {np.asarray(aux.tvalues)[1]:+.2f}  t(x^2) {np.asarray(aux.tvalues)[2]:+.2f}  "
      f"nR2 {len(x)*aux.rsquared:.1f} vs chi2(2) 5% crit 5.99")
plt.scatter(x, np.abs(e), s=4, alpha=0.3); plt.xlabel("market return"); plt.ylabel("|residual|")
plt.title("The fan: |e| against x"); plt.show()

**Expected pattern.** On real daily data the |x| quartiles' var(e) rises
noticeably (big market days are high-vol days for everything) and the χ²
smashes 5.99: **reject homoskedasticity, decisively, on the most standard
regression in finance.** On the synthetic pair the generator's own vol
clustering produces a milder fan — visible in time buckets more than in |x|
(time-varying variance less tied to |x|). That contrast is a finding, not a
grain of salt: the *form* of the heteroskedasticity determines how loud the
alarm rings, and White's estimator needs no knowledge of the form — but
your diagnostics should always check both |x|-shape and time-shape.

## E4 — both flavors across ten names

In [ ]:
if DATA_SOURCE == "real":
    from qrc.universe import load_universe
    tickers = load_universe("research50")[:10]
    pxr = get_prices(tickers + ["SPY"], start="2015-01-01")
    rets = np.log(pxr).diff().dropna()
    pairs = [(t, rets[t].values) for t in tickers]; mkt = rets["SPY"].values
else:
    from qrc.synth import synthetic_returns
    rw = synthetic_returns(n_days=2500, n_assets=11, seed=17, corr=0.45,
                           drift_spread=0.0001)
    rw.columns = ["SPY"] + [f"S{i}" for i in range(1, 11)]
    rets = rw
    pairs = [(t, rw[t].values) for t in rw.columns[1:]]; mkt = rw["SPY"].values

rows = []
for t, yv in pairs:
    g = ols_fit(mkt, yv)
    hc1 = se_white(g["X"], g["resid"])[1] * np.sqrt(len(mkt) / (len(mkt) - 2))
    rows.append([t, g["beta"], g["se_beta"], hc1, hc1 / g["se_beta"]])
tab = pd.DataFrame(rows, columns=["name", "beta", "classical SE", "White SE", "ratio"])
print(tab.round(4).to_string(index=False))
print(f"\nratio: mean {tab['ratio'].mean():.3f}   min {tab['ratio'].min():.3f}   max {tab['ratio'].max():.3f}")

**Expected pattern.** Ratios center near 1 but scatter ±10–20% in both
directions — the classical SE's error in the market model is
*direction-random*, not a systematic understatement: because here the
variance variation is partly time-regimes (classical ≈ fine on average),
partly |x|-loading (classical too small), partly tail days (classical too
large). **The meta-research consequence: you cannot 'correct' a
literature's t-stats by one global rule — the wrongness of unlabeled SEs is
itself heterogeneous, which is why module 13 treats reported t-stats as
*upper bounds on themselves*.** Common mistake: reporting the flavor whose
t crossed 2. That is SE-shopping, and it is a multiple-testing violation
(module 13 will name it).

## E5 — the repair boundary (exemplar)

The proposal repairs A4 (the fan — White's diagonal meat handles any σ_t²
pattern) and leaves A5 **broken**: 21-day overlapping observations make
ε_t an MA(20) *by construction*, so E[e_t e_{t−l}] ≠ 0 for l ≤ 20, and no
diagonal meat can represent that — White's estimator stays inconsistent for
Var(β̂) exactly as classical was. Day 8 extends the sandwich with the
off-diagonal lag terms (Newey–West): same bread, a meat that includes
e_t e_{t−l} cross-products with decaying weights, lag length ≥ the overlap.
The grade: **right estimator for one of your two violations; the sentence
"robust SEs" must carry the qualifier "heteroskedasticity-robust only;
autocorrelation unaddressed — see HAC."**

**Common mistakes across the day.** Believing White changes β̂; believing
White handles autocorrelation; treating a White−classical gap as a bug —
the gap is the diagnostic (a large, systematic gap says the classical
table was lying); and the professional sin: quoting the smaller SE because
it reads better.